In [41]:
import json
import time
from collections import Counter,defaultdict
from concurrent.futures import ProcessPoolExecutor
import multiprocessing

In [42]:
class BytePreTokenizer:
  """
     Splits text into individual UTF-8 bytes
  """

  def pre_tokenize(self,text):
     return [bytes([b]) for b in text.encode("utf-8")]

In [43]:
# PARALLEL PAIR COUNTING

def count_pairs_chunk(word_freq_chunk):
    """
      Counts adjacent byte-token pairs inside a chunk of word frequencies
    """
    pairs=defaultdict(int)
    for word,freq in word_freq_chunk.items():
      for i in range(len(word)-1):
        pairs[(word[i],word[i+1])] += freq
    return pairs

In [44]:
class BPETrainer:
  def __init__(self,vocab_size=300):
    self.vocab_size=vocab_size
    self.merges={}
    self.vocab={}

  def train(self,corpus,pretokenizer):
    if not corpus:
      raise ValueError("Corpus cannot be empty.")
    word_freqs = Counter()

    #Build initial word frequencies

    for text in corpus:
      tokens=pretokenizer.pre_tokenize(text)
      if not tokens:
        continue
      word = tuple(tokens)
      word_freqs[word] += 1

    # The following block is now correctly outside the initial word frequency loop
    if not word_freqs:
      raise ValueError("Corpus produced no tokens.")

    vocab = set()
    for word in word_freqs:
      vocab.update(word)

    merges={}
    merge_rank=0

    while len(vocab) < self.vocab_size:
      if len(word_freqs) == 0:
        break
      items = list(word_freqs.items())
      num_workers=multiprocessing.cpu_count()
      chunk_size = max(1,len(items) // num_workers)
      chunks = [word_freqs]

      with ProcessPoolExecutor() as executor:
        results=executor.map(count_pairs_chunk,chunks)
      pair_counts = defaultdict(int)
      for result in results:
        for pair,freq in result.items():
          pair_counts[pair] += freq
      if not pair_counts:
        break
      #select the most frequent pair
      best_pair = max(pair_counts,key=pair_counts.get)
      merges[best_pair]=merge_rank
      merge_rank += 1

      #Apply merge(TRUE BYTE CONCATENATION)
      new_word_freqs = {}
      for word,freq in word_freqs.items():
        new_word = []
        i=0

        while(i<len(word)):
          if i < len(word) - 1 and (word[i],word[i+1]) == best_pair:
            merged_token=word[i]+word[i+1]
            new_word.append(merged_token)
            i+=2
          else:
            new_word.append(word[i])
            i+=1
        new_word_freqs[tuple(new_word)] = freq
      word_freqs = new_word_freqs
      vocab.add(best_pair[0]+best_pair[1]) # Corrected 'best[1]' to 'best_pair[1]'

      # Build deterministic vocab mapping
      self.vocab = { token: idx for idx, token in enumerate(sorted(vocab)) }
      self.merges = merges

In [48]:
import json

class BPETokenizer:
  def __init__(self,trainer):
    self.vocab=trainer.vocab
    self.merges=trainer.merges
    self.reverse_vocab={
        idx: token for token,idx in self.vocab.items()
    }

  def encode(self, text):
    word = [bytes([b]) for b in text.encode("utf-8")]
    while True:
        if len(word) < 2:
            break

        pairs = [(word[i], word[i + 1]) for i in range(len(word) - 1)]

        # If there are no pairs, we can't merge anything further
        if not pairs:
            break

        candidate_pairs = [(pair, self.merges.get(pair, float("inf"))) for pair in pairs]

        # Filter out pairs not in self.merges (i.e., rank is float("inf"))
        mergable_pairs = [cp for cp in candidate_pairs if cp[1] != float("inf")]

        if not mergable_pairs: # No more merges left that are in self.merges
            break

        best_pair, best_rank = min(mergable_pairs, key=lambda x: x[1])

        # Apply merge
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == best_pair:
                new_word.append(word[i] + word[i + 1])
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        word = new_word
    return [self.vocab[token] for token in word]

  def decode(self, token_ids):
    byte_sequence = b''.join(self.reverse_vocab[token_id] for token_id in token_ids)
    return byte_sequence.decode("utf-8")

  def save(self, path):
    with open(path, "w") as f:
      # Convert tuple keys (bytes_a, bytes_b) into a single string key "str_a\u001fstr_b"
      serializable_merges = {
          f"{a.decode('latin1')}\u001f{b.decode('latin1')}": rank
          for (a, b), rank in self.merges.items()
      }
      json.dump({
          "vocab": { token.decode("latin1"): idx for token, idx in self.vocab.items() },
          "merges": serializable_merges
      }, f)

  @staticmethod
  def load(path):
    with open(path, "r") as f:
        data = json.load(f)
    trainer = BPETrainer() # Assuming BPETrainer is accessible here
    trainer.vocab = {token.encode("latin1"): idx for token, idx in data["vocab"].items()}

    # Reconstruct merges: parse string key "str_a\u001fstr_b" back to (bytes_a, bytes_b)
    reconstructed_merges = {}
    for key_str, rank in data["merges"].items():
        parts = key_str.split('\u001f')
        if len(parts) == 2:
            reconstructed_merges[(parts[0].encode("latin1"), parts[1].encode("latin1"))] = rank
    trainer.merges = reconstructed_merges
    return BPETokenizer(trainer)

In [46]:
def benchmark(tokenizer, text):
    start = time.time()
    encoded = tokenizer.encode(text)
    end = time.time()
    print("Encoded length:", len(encoded))
    print("Time taken:", end - start)

In [49]:
if __name__ == "__main__":
    corpus = ["low lower lowest", "newer wider", "low low low"]
    pretokenizer = BytePreTokenizer()
    trainer = BPETrainer(vocab_size=100)
    trainer.train(corpus, pretokenizer)
    tokenizer = BPETokenizer(trainer)
    text = "lowest"
    encoded = tokenizer.encode(text)
    decoded = tokenizer.decode(encoded)
    print("Original:", text)
    print("Encoded:", encoded)
    print("Decoded:", decoded)
    benchmark(tokenizer, text)
    tokenizer.save("bpe_tokenizer.json")

Original: lowest
Encoded: [8, 3, 27, 28]
Decoded: lowest
Encoded length: 4
Time taken: 2.86102294921875e-05
